# Wait-Injection Mechanism Spot-Check (real model, real MATH-500 items)

**What this notebook does, in plain terms:** the two Wait-injection functions (`generate_capped_no_injection` for short, `generate_stop_triggered_wait` for long) have ONLY ever been tested against a fake, scripted model. They've never touched the real model. This notebook runs them on a handful of real MATH-500 problems and checks, by actually reading the transcripts, whether the mechanism does what it's supposed to:

- **Short condition:** clean early answer, zero Wait injections.
- **Long condition:** the model tries to stop, gets intercepted, says "Wait" (or our fuller phrase), and genuinely reconsiders, rather than just padding with repeated text.

**This ALSO runs both Wait phrasings side by side** ("Wait" -- s1's literal wording -- vs "Wait, let me double-check that." -- our fuller phrase) on the same items, so we get real data on whether the wording matters, not just a plumbing check.

**This is NOT the full drift run.** It's 2-3 items, single GPU, meant to be read carefully by a human before we trust the mechanism at scale.

**Files to upload into this folder first:** `harness.py` (v0.3.4), `wait_injection.py`, `math500_matcher.py`. Run top to bottom.

## 1. Install packages
Let this finish completely before running anything else.

In [ ]:
!pip install -q numpy torch transformers accelerate datasets


## 2. Version check -- FIRST, every time

Confirms the on-disk files have what this notebook needs before any GPU time is spent.

In [ ]:
import harness, wait_injection, math500_matcher, inspect
print("harness version:", harness.HARNESS_VERSION, "(want 0.3.4)")
print("wait_injection has generate_capped_no_injection:",
      hasattr(wait_injection, "generate_capped_no_injection"))
print("wait_injection has generate_stop_triggered_wait:",
      hasattr(wait_injection, "generate_stop_triggered_wait"))
print("wait_injection has both phrasing constants:",
      hasattr(wait_injection, "WAIT_TEXT_S1_LITERAL"), hasattr(wait_injection, "WAIT_TEXT_FULLER"))
print("  WAIT_TEXT_S1_LITERAL =", repr(wait_injection.WAIT_TEXT_S1_LITERAL))
print("  WAIT_TEXT_FULLER     =", repr(wait_injection.WAIT_TEXT_FULLER))

import torch
print("GPU available:", torch.cuda.is_available())


## 3. Load the model

Single GPU is plenty for a 2-3 item spot-check, no need for the multi-GPU worker machinery here.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda:0"
)
model.eval()
print("loaded on:", model.device)


## 4. Pick the spot-check items

3 MATH-500 problems, fixed seed so this is reproducible. Deliberately small -- this is meant to be READ, not just measured.

In [ ]:
import random
from datasets import load_dataset
from math500_matcher import extract_math500_gold

ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
idxs = list(range(len(ds)))
random.Random(42).shuffle(idxs)
spot_idxs = idxs[:3]

spot_items = [(i, ds[i]["problem"], extract_math500_gold(ds[i]), ds[i]["level"]) for i in spot_idxs]
for i, prob, gold, lvl in spot_items:
    print(f"idx={i} level={lvl} gold={gold!r}")
    print(f"   problem: {prob[:150]}...")


## 5. Run all three conditions on each item

For each item: short (no injection), long with "Wait" (s1 literal), long with "Wait, let me double-check that." (fuller). Budgets are modest for a spot-check, we're reading transcripts, not measuring accuracy at scale yet.

**This will take a while** (each long-condition generation can run into the hundreds of seconds), that's expected for 3 items x 3 conditions.

In [ ]:
from wait_injection import (generate_capped_no_injection, generate_stop_triggered_wait,
                            WAIT_TEXT_S1_LITERAL, WAIT_TEXT_FULLER)
from math500_matcher import extract_math500_pred, is_equiv_math
import time, json

SHORT_BUDGET = 1024
LONG_BUDGET = 8192

spot_results = []

for i, prob, gold, lvl in spot_items:
    print(f"\n{'='*70}\nidx={i} (level {lvl})\n{'='*70}")

    # -- short: no injection, ceiling only --
    t0 = time.time()
    text_s, think_s, marg_s, ent_s, full_s, closed_naturally = generate_capped_no_injection(
        model, tokenizer, prob, budget=SHORT_BUDGET, seed=0)
    dt_s = time.time() - t0
    pred_s = extract_math500_pred(text_s)
    correct_s, tier_s = is_equiv_math(pred_s, gold)
    print(f"SHORT: closed_naturally={closed_naturally} think_tokens={think_s} "
          f"pred={pred_s!r} correct={correct_s} ({dt_s:.0f}s)")

    # -- long, s1 literal "Wait" --
    t0 = time.time()
    text_lit, think_lit, marg_lit, ent_lit, full_lit, n_inj_lit, offsets_lit = generate_stop_triggered_wait(
        model, tokenizer, prob, long_budget=LONG_BUDGET, seed=0, wait_text=WAIT_TEXT_S1_LITERAL)
    dt_lit = time.time() - t0
    pred_lit = extract_math500_pred(text_lit)
    correct_lit, tier_lit = is_equiv_math(pred_lit, gold)
    print(f"LONG (\"Wait\"): n_injections={n_inj_lit} offsets={offsets_lit} "
          f"think_tokens={think_lit} pred={pred_lit!r} correct={correct_lit} ({dt_lit:.0f}s)")

    # -- long, our fuller phrase --
    t0 = time.time()
    text_full, think_full, marg_full, ent_full, full_full, n_inj_full, offsets_full = generate_stop_triggered_wait(
        model, tokenizer, prob, long_budget=LONG_BUDGET, seed=0, wait_text=WAIT_TEXT_FULLER)
    dt_full = time.time() - t0
    pred_full = extract_math500_pred(text_full)
    correct_full, tier_full = is_equiv_math(pred_full, gold)
    print(f"LONG (fuller): n_injections={n_inj_full} offsets={offsets_full} "
          f"think_tokens={think_full} pred={pred_full!r} correct={correct_full} ({dt_full:.0f}s)")

    spot_results.append(dict(
        idx=i, level=lvl, gold=gold,
        short=dict(text=text_s, think_tokens=think_s, closed_naturally=closed_naturally,
                   pred=pred_s, correct=correct_s, wall_clock_sec=round(dt_s,1)),
        long_literal=dict(text=text_lit, think_tokens=think_lit, n_injections=n_inj_lit,
                          injection_offsets=offsets_lit, pred=pred_lit, correct=correct_lit,
                          wall_clock_sec=round(dt_lit,1)),
        long_fuller=dict(text=text_full, think_tokens=think_full, n_injections=n_inj_full,
                         injection_offsets=offsets_full, pred=pred_full, correct=correct_full,
                         wall_clock_sec=round(dt_full,1)),
    ))

with open("wait_spotcheck.json", "w") as f:
    json.dump(spot_results, f, indent=2)
print("\n\nSaved to wait_spotcheck.json")


## 6. Read the actual transcripts

This is the part that matters most. Numbers alone (n_injections, correct/incorrect) don't tell you whether the mechanism produced GENUINE reconsideration or just padding/repetition. Read the text around each injection point. Look for:
- Does the model actually re-examine its work after "Wait," or does it just restate the same thing?
- Does the short condition give a clean, confident answer, not a truncated fragment?
- Does either phrasing look different in kind, not just in the words themselves?

In [ ]:
for r in spot_results:
    print(f"\n{'#'*70}\nidx={r['idx']} (level {r['level']}, gold={r['gold']!r})\n{'#'*70}")

    print(f"\n--- SHORT (budget={SHORT_BUDGET}, closed_naturally={r['short']['closed_naturally']}) ---")
    print(f"pred={r['short']['pred']!r} correct={r['short']['correct']}")
    print("...tail:", r['short']['text'][-400:])

    print(f"\n--- LONG, \"Wait\" ({r['long_literal']['n_injections']} injections) ---")
    print(f"pred={r['long_literal']['pred']!r} correct={r['long_literal']['correct']}")
    print("...tail:", r['long_literal']['text'][-400:])

    print(f"\n--- LONG, fuller phrase ({r['long_fuller']['n_injections']} injections) ---")
    print(f"pred={r['long_fuller']['pred']!r} correct={r['long_fuller']['correct']}")
    print("...tail:", r['long_fuller']['text'][-400:])


## 7. Quick summary table

A compact view before deciding if the mechanism is trustworthy enough to run at scale.

In [ ]:
print(f"{'idx':>5} {'lvl':>3}  {'short ok':>8}  {'Wait ok/#inj':>14}  {'fuller ok/#inj':>16}")
for r in spot_results:
    s = "Y" if r['short']['correct'] else "N"
    l = f"{'Y' if r['long_literal']['correct'] else 'N'}/{r['long_literal']['n_injections']}"
    f_ = f"{'Y' if r['long_fuller']['correct'] else 'N'}/{r['long_fuller']['n_injections']}"
    print(f"{r['idx']:>5} {r['level']:>3}  {s:>8}  {l:>14}  {f_:>16}")
